# Skillova Trainee Assessment - Category 5: Data Analysis

## Objective
Clean a messy dataset of 1,000 customer-support tickets, then answer:

1. What is the average resolution time after cleaning?
2. Which ticket category takes the longest to solve?

The dataset contains blank resolution times and extreme values such as 999,999 hours.

## 1. Load and inspect the data

Before calculating results, inspect the row count, data types, missing values, and resolution-time statistics.

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/customer_support_tickets_messy.csv")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()

,Ticket_ID,Category,Resolution_Time_Hours
0,TKT-0001,Billing,26.44
1,TKT-0002,Billing,7.00
2,TKT-0003,General Inquiry,3.38
3,TKT-0004,Technical,23.79
4,TKT-0005,Technical,45.92


In [2]:
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nResolution-time statistics before cleaning:")
print(df["Resolution_Time_Hours"].describe())

Rows: 1,000
Columns: 3

Data types:
Ticket_ID                 object
Category                  object
Resolution_Time_Hours    float64
dtype: object

Missing values:
Ticket_ID                 0
Category                  0
Resolution_Time_Hours    40
dtype: int64

Resolution-time statistics before cleaning:
count       960.000000
mean      10434.089219
std      101580.138597
min           0.710000
25%           7.335000
50%          14.450000
75%          24.425000
max      999999.000000
Name: Resolution_Time_Hours, dtype: float64


## 2. Fix blank resolution times

Blank resolution times are filled with the median for the same ticket category.

The median is used because it is less affected by extreme values than the mean. A category-level median is more appropriate than one overall median because ticket categories have different normal resolution times.

In [3]:
cleaning_df = df.copy()
cleaning_df["Resolution_Time_Hours"] = pd.to_numeric(
    cleaning_df["Resolution_Time_Hours"], errors="coerce"
)

missing_before = cleaning_df["Resolution_Time_Hours"].isna().sum()
category_medians = cleaning_df.groupby("Category")["Resolution_Time_Hours"].transform("median")
cleaning_df["Resolution_Time_Hours"] = cleaning_df["Resolution_Time_Hours"].fillna(category_medians)
missing_after = cleaning_df["Resolution_Time_Hours"].isna().sum()

print(f"Blank values before fixing: {missing_before}")
print(f"Blank values after fixing: {missing_after}")

Blank values before fixing: 40
Blank values after fixing: 0


## 3. Identify and remove extreme outliers

A conservative IQR rule is used. A value is treated as an extreme high outlier when it is above:

`Q3 + 3 x IQR`

Using 3 x IQR instead of 1.5 x IQR avoids removing valid tickets that simply took longer than usual.

In [4]:
q1 = cleaning_df["Resolution_Time_Hours"].quantile(0.25)
q3 = cleaning_df["Resolution_Time_Hours"].quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + (3 * iqr)

extreme_outlier_mask = cleaning_df["Resolution_Time_Hours"] > upper_limit
extreme_outliers = cleaning_df.loc[extreme_outlier_mask]
clean_df = cleaning_df.loc[~extreme_outlier_mask].copy()

print(f"Q1: {q1:.2f} hours")
print(f"Q3: {q3:.2f} hours")
print(f"IQR: {iqr:.2f} hours")
print(f"Extreme-outlier upper limit: {upper_limit:.2f} hours")
print(f"Extreme outliers removed: {len(extreme_outliers)}")
print(f"Rows after cleaning: {len(clean_df):,}")
print("\nExtreme values found:")
print(extreme_outliers["Resolution_Time_Hours"].value_counts())

Q1: 7.33 hours
Q3: 24.47 hours
IQR: 17.14 hours
Extreme-outlier upper limit: 75.88 hours
Extreme outliers removed: 10
Rows after cleaning: 990

Extreme values found:
Resolution_Time_Hours
999999.0    10
Name: count, dtype: int64


## 4. Calculate the required insights

In [5]:
average_resolution_time = clean_df["Resolution_Time_Hours"].mean()

category_summary = (
    clean_df.groupby("Category", as_index=False)
    .agg(
        Ticket_Count=("Ticket_ID", "count"),
        Average_Resolution_Time_Hours=("Resolution_Time_Hours", "mean"),
    )
    .sort_values("Average_Resolution_Time_Hours", ascending=False)
)

category_summary["Average_Resolution_Time_Hours"] = category_summary[
    "Average_Resolution_Time_Hours"
].round(2)

print(f"Average resolution time after cleaning: {average_resolution_time:.2f} hours")
category_summary

Average resolution time after cleaning: 17.57 hours


,Category,Ticket_Count,Average_Resolution_Time_Hours
4,Technical,278,28.83
3,Product Issue,209,21.97
1,Billing,201,13.95
0,Account Access,158,7.92
2,General Inquiry,144,5.08


In [6]:
longest_category = category_summary.iloc[0]
print(
    f"The category that takes the longest to solve is "
    f"{longest_category['Category']} at "
    f"{longest_category['Average_Resolution_Time_Hours']:.2f} hours on average."
)

The category that takes the longest to solve is Technical at 28.83 hours on average.


## 5. Export the cleaned files

In [7]:
clean_df.to_csv(OUTPUT_DIR / "customer_support_tickets_cleaned.csv", index=False)
category_summary.to_csv(OUTPUT_DIR / "category_resolution_summary.csv", index=False)

print("Exported cleaned dataset and category summary.")

Exported cleaned dataset and category summary.


## Conclusion

After cleaning, the average resolution time is **17.57 hours**. **Technical** tickets take the longest to solve, at **28.83 hours** on average.

The cleaning process fixed 40 blank values using category medians and removed 10 extreme values of 999,999 hours.